In [1]:
!pip install transformers torch sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 55.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"  # Chọn phiên bản phù hợp với GPU của bạn (base, large, xl, xxl)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [103]:
# essay = """
# The rapid development of artificial intelligence (AI) has revolutionized many industries. [MISSING]. However, ethical concerns about data privacy remain unresolved.
# """
# essay = "Neural networks demonstrate remarkable pattern recognition capabilities. [MISSING]. This makes them ideal for medical image analysis."

def gen_prompt(essay):
  prompt = f"""
  Task: Generate a coherent sentence to fill in the [MISSING] gap within an academic essay. The sentence must:
  1. Logically connect the preceding and following context
  2. Maintain formal academic tone
  3. Be 15-25 words long
  4. Include relevant keywords from the surrounding text
  Current Essay Context: {essay}
  Generated Sentence:
  """
  return prompt

In [104]:
import pandas as pd
import nltk
import random
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [105]:
df = pd.read_csv('ielts_dataset_v1.4_rawxparaphrased.csv')
df["replace_index"] = [None] * len(df)
df.head()

,prompt,text,is_ai,variant,replace_index
0,In many countries children are engaged in some...,"In many countries, the issue of children engag...",1,raw,None
1,News editors decide what to broadcast on telev...,The role of news editors in determining what i...,1,raw,None
2,Popular events like the football World Cup and...,The assertion that popular events such as the ...,1,raw,None
3,In many countries children are engaged in some...,"In many countries, the issue of children engag...",1,raw,None
4,In many countries children are engaged in some...,"In many countries, the involvement of children...",1,raw,None


In [106]:
def replacement_1(essay, mask_ratio=0.2):
  sentences = nltk.sent_tokenize(essay)
  num_to_mask = max(1, int(len(sentences) * mask_ratio))
  masked_indices = random.sample(range(len(sentences)), num_to_mask)
  for index in masked_indices:
    sentences[index] = "[MISSING]."
    input_essay = " ".join(sentences)
    new_sentence = gen_text(essay)
    print(f"{index} : {new_sentence}")
    sentences[index] = new_sentence

  return masked_indices, " ".join(sentences)

In [107]:
def replacement_2(essay, mask_ratio=0.2):
  sentences = nltk.sent_tokenize(essay)
  n = random.sample(range(5), 1)[0]
  masked_indices = [n + i for i in range(3)]

  for index in masked_indices:
    sentences[index] = "[MISSING]."
    input_essay = " ".join(sentences)
    new_sentence = gen_text(essay)
    print(f"{index} : {new_sentence}")
    sentences[index] = new_sentence

  return masked_indices, " ".join(sentences)

In [108]:
def gen_text(essay):
  input_text = gen_prompt(essay)
  inputs = tokenizer(
      input_text,
      return_tensors="pt",
      max_length=512,
      truncation=True,
      padding="max_length"
  )

  outputs = model.generate(
      input_ids=inputs.input_ids,
      attention_mask=inputs.attention_mask,
      do_sample=True,
      max_new_tokens=100,
      temperature=1.3,
      num_beams=5,
      early_stopping=True
  )

  result = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return result

In [109]:
# Main
for index, row in df.iterrows():
  if(row["is_ai"] == 1 and row["variant"] == 'raw'):
    essay = row["text"]
    a, b  = replacement_1(essay)
    df.at[index, "replace_index"] = a
    df.at[index, "text"] = b
    break
  elif(row["is_ai"] == 0):
    essay = row["text"]
    a, b  = replacement_2(essay)
    df.at[index, "replace_index"] = a
    df.at[index, "text"] = b
    df.at[index, "variant"] = "word-sub-huai"
    df.at[index, "is_ai"] = 2
  else:
    continue
df.head()

7 : Regardless of your perspective on the issue of child labor, there are valid concerns about the potential negative impacts on education and childhood well-being.
9 : In many countries, the issue of children engaging in paid work has sparked considerable debate.


,prompt,text,is_ai,variant,replace_index
0,In many countries children are engaged in some...,"In many countries, the issue of children engag...",1,raw,"[7, 9]"
1,News editors decide what to broadcast on telev...,The role of news editors in determining what i...,1,raw,None
2,Popular events like the football World Cup and...,The assertion that popular events such as the ...,1,raw,None
3,In many countries children are engaged in some...,"In many countries, the issue of children engag...",1,raw,None
4,In many countries children are engaged in some...,"In many countries, the involvement of children...",1,raw,None
